# Datasets and DataLoaders
---

## Loading a Dataset
> Here we download a sample dataset.

In [3]:
import torch
from torchvision import datasets
from torch.utils.data import DataLoader

from torchvision.transforms import v2

In [7]:
sample_dataset = datasets.FashionMNIST(
    root="./data",
    train=True, download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(dtype=torch.float32, scale=True)])
)

In [9]:
sample_dataset

Dataset FashionMNIST
    Number of datapoints: 60000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
                 ToImage()
                 ToDtype(scale=True)
           )

### Exploring a Dataset

In [10]:
len(sample_dataset)

60000

In [22]:
for X,y in sample_dataset:
  print(type(X), type(y))
  print(X.shape, y)
  break

<class 'torchvision.tv_tensors._image.Image'> <class 'int'>
torch.Size([1, 28, 28]) 9


## Creating Custom Datasets
> A custom dataset must implement the following `__init__`, `__len__` and `__getitem__`

**Note**
This uses the `/content/sample_data` of the google collab environment.

**Goal**
The goal is to create a custom dataset class from existing data.

### EDA

In [24]:
from pathlib import Path
DATA_PATH = Path("/content/sample_data/")
TRAIN_DATA_RAW = DATA_PATH/"california_housing_train.csv"
TEST_DATA_RAW = DATA_PATH/"california_housing_test.csv"

In [27]:
import pandas as pd

train_df = pd.read_csv(TEST_DATA_RAW)
train_df

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
0,-122.05,37.37,27.0,3885.0,661.0,1537.0,606.0,6.6085,344700.0
1,-118.30,34.26,43.0,1510.0,310.0,809.0,277.0,3.5990,176500.0
2,-117.81,33.78,27.0,3589.0,507.0,1484.0,495.0,5.7934,270500.0
3,-118.36,33.82,28.0,67.0,15.0,49.0,11.0,6.1359,330000.0
4,-119.67,36.33,19.0,1241.0,244.0,850.0,237.0,2.9375,81700.0
...,...,...,...,...,...,...,...,...,...
2995,-119.86,34.42,23.0,1450.0,642.0,1258.0,607.0,1.1790,225000.0
2996,-118.14,34.06,27.0,5257.0,1082.0,3496.0,1036.0,3.3906,237200.0
2997,-119.70,36.30,10.0,956.0,201.0,693.0,220.0,2.2895,62000.0
2998,-117.12,34.10,40.0,96.0,14.0,46.0,14.0,3.2708,162500.0


In [31]:
print(train_df.columns)
print(train_df.describe())

Index(['longitude', 'latitude', 'housing_median_age', 'total_rooms',
       'total_bedrooms', 'population', 'households', 'median_income',
       'median_house_value'],
      dtype='object')
         longitude    latitude  housing_median_age   total_rooms  \
count  3000.000000  3000.00000         3000.000000   3000.000000   
mean   -119.589200    35.63539           28.845333   2599.578667   
std       1.994936     2.12967           12.555396   2155.593332   
min    -124.180000    32.56000            1.000000      6.000000   
25%    -121.810000    33.93000           18.000000   1401.000000   
50%    -118.485000    34.27000           29.000000   2106.000000   
75%    -118.020000    37.69000           37.000000   3129.000000   
max    -114.490000    41.92000           52.000000  30450.000000   

       total_bedrooms    population  households  median_income  \
count     3000.000000   3000.000000  3000.00000    3000.000000   
mean       529.950667   1402.798667   489.91200       3.807272  

In [74]:
data = train_df.to_numpy()
data[1, 8:9].shape

(1,)

### Define Custom Dataset
> We should create an implementaiton of __len__, __getitem__, and __init__

In [32]:
from torch.utils.data import Dataset, DataLoader

In [98]:
class HousingDataset(Dataset):
  def __init__(self, root: Path, train: bool = True, transform=None, target_transform=None):
    # we load the data already at __init__, we can perform lazy loading here and execute actual loading in getitem if necessary
    print(root/f"california_housing_{'train' if train else 'test'}.csv")
    self.data = pd.read_csv(root/f"california_housing_{'train' if train else 'test'}.csv").to_numpy()
    self.transform = transform
    self.target_transform = target_transform

  def __len__(self):
    return len(self.data)

  def __getitem__(self, idx):
    X, y =self. data[idx, :9], self.data[idx, 8:9]
    if self.transform:
      X = self.transform(X)
    if self.target_transform:
      y = self.transform(y)
    return X,y


In [102]:
# note we can pass on some transform like scaler to perform scaling on the data, for our simple case we dont need to.
train_dataset = HousingDataset(DATA_PATH, train=True)
train_dataset

/content/sample_data/california_housing_train.csv


In [103]:
test_dataset = HousingDataset(DATA_PATH, train=False)
print(len(test_dataset))
print(test_dataset)

/content/sample_data/california_housing_test.csv
3000


In [104]:
print(len(train_dataset))
for idx, (X, y) in enumerate(train_dataset):
  print(idx)
  print(X.shape, X)
  print(y.shape, y)
  break

17000
0
(9,) [-1.1431e+02  3.4190e+01  1.5000e+01  5.6120e+03  1.2830e+03  1.0150e+03
  4.7200e+02  1.4936e+00  6.6900e+04]
(1,) [66900.]


## Creating a Custom Dataloader

In [105]:
train_dataloader = DataLoader(dataset=train_dataset, batch_size=32, shuffle=True)

In [113]:
X,y = next(iter(train_dataloader))
print(X.shape, y.shape)

torch.Size([32, 9]) torch.Size([32, 1])


In [115]:
for X,y in train_dataloader:
  print(X.shape, y.shape)
  break

torch.Size([32, 9]) torch.Size([32, 1])


## TODO
> In the next session, we'll create a `generic neural network predictor` that predicts the median house value given features.